# Chinese Speed Limit Signs -- YOLOv5n Training (TT100K)

Train an 8-class YOLOv5n model for the AI-Powered HUD project (China market).

**Data source:** Tsinghua-Tencent 100K (TT100K)

**Target device:** Luckfox Pico Ultra (RV1106G3, 0.5 TOPS NPU, INT8)

| ID | Class | TT100K Label | ID | Class | TT100K Label |
|----|-------|--------------|----|-------|--------------|
| 0 | speed_sign_20 | pl20 | 4 | speed_sign_60 | pl60 |
| 1 | speed_sign_30 | pl30 | 5 | speed_sign_80 | pl80 |
| 2 | speed_sign_40 | pl40 | 6 | speed_sign_100 | pl100 |
| 3 | speed_sign_50 | pl50 | 7 | speed_sign_120 | pl120 |

**Runtime:** Change to GPU (Runtime > Change runtime type > T4 GPU)

## 1. Environment Setup

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# Clone airockchip/yolov5 (RKNN-optimized fork, required for --rknpu export)
!git clone https://github.com/airockchip/yolov5.git
%cd yolov5
!pip install -r requirements.txt -q
!pip install pyyaml tqdm -q

# [Fix] onnxscript is required by ONNX export with PyTorch >= 2.6
!pip install onnxscript -q

# [Fix] Pillow 10+ removed font.getsize() used by YOLOv5 utils/plots.py
# Pin to Pillow 9.x to avoid AttributeError during val/detect visualization
!pip install "Pillow<10" -q

## 2. Download & Prepare TT100K Dataset

Download the TT100K dataset, filter for 8 speed-limit classes, and convert
to YOLO format using `cn_tt100k_prepare.py`.

The script will:
1. Download `data.zip` (~1.7 GB) from the official TT100K source
2. Parse `annotations.json` and filter for pl20/30/40/50/60/80/100/120
3. Convert bounding boxes to YOLO format (normalized cx, cy, w, h)
4. Split into 80% train / 20% val
5. Generate `data.yaml` config file

In [ ]:
# Upload cn_tt100k_prepare.py from the training/ directory
from google.colab import files
import os, shutil

print("Upload cn_tt100k_prepare.py from the training/ directory...")
uploaded = files.upload()

if uploaded:
    fname = list(uploaded.keys())[0]
    # Always copy to /content/ so it's accessible regardless of cwd
    dest = "/content/cn_tt100k_prepare.py"
    shutil.copy(fname, dest)
    print(f"Uploaded and copied to: {dest}")
else:
    print("[Error] No file uploaded.")

In [ ]:
# Download and prepare TT100K dataset
# This will download ~1.7 GB and create the YOLO-format dataset
#
# If you already have data.zip, upload it and use:
#   !python cn_tt100k_prepare.py --data-zip /content/data.zip --output /content/cn_speed_dataset

DATASET_ROOT = "/content/cn_speed_dataset"

!python /content/cn_tt100k_prepare.py \
    --output "{DATASET_ROOT}" \
    --downloads /content/downloads \
    --val-ratio 0.2 \
    --seed 42 \
    --clean

## 3. Dataset Statistics & Visualization

Verify the dataset was prepared correctly -- check class distribution and
visualize sample images with bounding boxes.

In [ ]:
import os, yaml
from collections import defaultdict
from pathlib import Path

DATASET_ROOT = "/content/cn_speed_dataset"

TARGET_CLASSES = [
    "speed_sign_20",  "speed_sign_30",  "speed_sign_40",  "speed_sign_50",
    "speed_sign_60",  "speed_sign_80",  "speed_sign_100", "speed_sign_120",
]

# Count annotations per class per split
print("=" * 65)
print("  Dataset Statistics")
print("=" * 65)

total_stats = defaultdict(int)

for split in ["train", "val"]:
    lbl_dir = f"{DATASET_ROOT}/{split}/labels"
    img_dir = f"{DATASET_ROOT}/{split}/images"

    if not os.path.exists(lbl_dir):
        print(f"  {split}: not found")
        continue

    n_images = len([f for f in os.listdir(img_dir)
                    if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    split_counts = defaultdict(int)

    for lbl_file in os.listdir(lbl_dir):
        if not lbl_file.endswith(".txt"):
            continue
        with open(os.path.join(lbl_dir, lbl_file), "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(parts[0])
                    split_counts[cls_id] += 1
                    total_stats[cls_id] += 1

    print(f"\n  {split}: {n_images} images")
    for cls_id in range(len(TARGET_CLASSES)):
        count = split_counts.get(cls_id, 0)
        print(f"    {cls_id}: {TARGET_CLASSES[cls_id]:<18s} {count:>5d}")

# Total summary
print(f"\n  {'Class':<5s} {'Name':<20s} {'Total':>7s}  Distribution")
print("  " + "-" * 55)
total_ann = sum(total_stats.values())
for cls_id in range(len(TARGET_CLASSES)):
    name = TARGET_CLASSES[cls_id]
    count = total_stats.get(cls_id, 0)
    pct = (count / total_ann * 100) if total_ann > 0 else 0
    bar = "#" * min(count // 10, 30)
    print(f"  {cls_id:<5d} {name:<20s} {count:>7d}  {bar}")
print("  " + "-" * 55)
print(f"  {'':5s} {'TOTAL':<20s} {total_ann:>7d}")

# Warn about low-data classes
for cls_id in range(len(TARGET_CLASSES)):
    count = total_stats.get(cls_id, 0)
    if count < 150:
        print(f"\n  [Warning] {TARGET_CLASSES[cls_id]} has only {count} instances -- "
              f"consider augmentation or additional data")

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import random

COLORS = [
    (255, 80, 80), (80, 255, 80), (80, 80, 255), (255, 255, 80),
    (255, 80, 255), (80, 255, 255), (255, 160, 80), (160, 80, 255),
]

def draw_yolo_boxes(img_path, label_path, class_names):
    """Draw YOLO bounding boxes on an image."""
    img = cv2.imread(img_path)
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    if os.path.exists(label_path):
        with open(label_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls_id = int(parts[0])
                cx, cy, bw, bh = [float(x) for x in parts[1:5]]

                x1 = int((cx - bw / 2) * w)
                y1 = int((cy - bh / 2) * h)
                x2 = int((cx + bw / 2) * w)
                y2 = int((cy + bh / 2) * h)

                color = COLORS[cls_id % len(COLORS)]
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                label = f"{class_names[cls_id]}" if cls_id < len(class_names) else f"cls{cls_id}"
                cv2.putText(img, label, (x1, y1 - 5),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    return img


# Show 8 random training samples
train_img_dir = f"{DATASET_ROOT}/train/images"
train_lbl_dir = f"{DATASET_ROOT}/train/labels"
all_imgs = [f for f in os.listdir(train_img_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

if all_imgs:
    samples = random.sample(all_imgs, min(8, len(all_imgs)))

    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    for ax, img_file in zip(axes.flat, samples):
        stem = Path(img_file).stem
        img = draw_yolo_boxes(
            os.path.join(train_img_dir, img_file),
            os.path.join(train_lbl_dir, stem + ".txt"),
            TARGET_CLASSES,
        )
        if img is not None:
            ax.imshow(img)
            ax.set_title(img_file[:30], fontsize=8)
        ax.axis("off")
    plt.suptitle("TT100K Training Samples with Bounding Boxes", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("No training images found. Check dataset preparation above.")

## 4. Train YOLOv5n

Transfer learning from COCO-pretrained weights. Training at 480x480 to better
detect small speed signs in TT100K's 2048x2048 source images.

Key differences from AU model:
- **imgsz=480** (AU used 320) -- TT100K signs are often small in high-res images
- **epochs=150** (AU used 100) -- more epochs for 8 classes with varied instance counts
- **8 classes** (AU had 9) -- no speed_camera class, but added pl20 and pl120

Expected training time: ~30-50 min on T4 GPU for 150 epochs.

**Compatibility fixes applied automatically:**
- PyTorch >= 2.6: `torch.load()` default changed to `weights_only=True`, patched to `False`
- Pillow 10+: `font.getsize()` removed, patched to `font.getbbox()` (if Pillow pin failed)

In [ ]:
%cd /content/yolov5

# ============================================================
# [Fix] PyTorch >= 2.6: torch.load() default weights_only=True
# breaks loading YOLOv5 .pt checkpoints.
#
# IMPORTANT: Do NOT use sed for this! sed regex cannot handle
# nested parentheses like torch.load(attempt_download(w), ...)
# which causes weights_only=False to be inserted into the wrong
# function call. Use Python-based patching instead.
# ============================================================
import subprocess

def patch_torch_load(filepath):
    """Add weights_only=False to all torch.load() calls, handling nested parens."""
    with open(filepath, 'r') as f:
        content = f.read()

    original = content
    result = []
    i = 0

    while i < len(content):
        idx = content.find('torch.load(', i)
        if idx == -1:
            result.append(content[i:])
            break

        result.append(content[i:idx])

        # Find matching closing paren (correctly handles nesting)
        start = idx + len('torch.load(')
        depth = 1
        j = start
        while j < len(content) and depth > 0:
            if content[j] == '(':
                depth += 1
            elif content[j] == ')':
                depth -= 1
            j += 1

        args_str = content[start:j-1]

        if 'weights_only' not in args_str:
            result.append(f'torch.load({args_str}, weights_only=False)')
        else:
            result.append(f'torch.load({args_str})')

        i = j

    content = ''.join(result)

    if content != original:
        with open(filepath, 'w') as f:
            f.write(content)
        return True
    return False


files_out = subprocess.run(
    ["grep", "-rl", "torch.load", "."],
    capture_output=True, text=True
).stdout.strip().split("\n")

patched = 0
for f in files_out:
    if f.endswith('.py'):
        if patch_torch_load(f):
            patched += 1
            print(f"  Patched: {f}")
print(f"torch.load patch: {patched} files fixed\n")

# ============================================================
# [Fix] Pillow 10+ removed font.getsize() in YOLOv5 plots.py
# Fallback patch in case "pip install Pillow<10" was overridden
# ============================================================
plots_py = "utils/plots.py"
with open(plots_py, 'r') as f:
    plots_content = f.read()

if 'getsize' in plots_content:
    plots_content = plots_content.replace(
        'w, h = self.font.getsize(label)',
        'try:\n                w, h = self.font.getsize(label)\n            except AttributeError:\n                bbox = self.font.getbbox(label); w, h = bbox[2] - bbox[0], bbox[3] - bbox[1]'
    )
    with open(plots_py, 'w') as f:
        f.write(plots_content)
    print("Pillow getsize patch: utils/plots.py fixed\n")

# ============================================================
# Update data.yaml paths for Colab
# ============================================================
import yaml

DATASET_ROOT = "/content/cn_speed_dataset"

data_yaml = {
    "train": f"{DATASET_ROOT}/train/images",
    "val": f"{DATASET_ROOT}/val/images",
    "nc": 8,
    "names": [
        "speed_sign_20",  "speed_sign_30",  "speed_sign_40",  "speed_sign_50",
        "speed_sign_60",  "speed_sign_80",  "speed_sign_100", "speed_sign_120",
    ],
}

data_yaml_path = f"{DATASET_ROOT}/data.yaml"
with open(data_yaml_path, "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)
print(f"Dataset config updated: {data_yaml_path}\n")

# ============================================================
# Train YOLOv5n with transfer learning
# ============================================================
!python train.py \
    --data "{DATASET_ROOT}/data.yaml" \
    --cfg yolov5n.yaml \
    --weights yolov5n.pt \
    --img 480 \
    --batch-size 32 \
    --epochs 150 \
    --workers 2 \
    --project runs/cn_speed_signs \
    --name v1 \
    --exist-ok \
    --cache ram

## 5. Evaluate Results

View training curves, confusion matrix, and run validation.

In [ ]:
# Show training curves
from IPython.display import Image, display
import os

results_dir = "/content/yolov5/runs/cn_speed_signs/v1"

# Training curves
results_img = f"{results_dir}/results.png"
if os.path.exists(results_img):
    display(Image(filename=results_img, width=900))

# Confusion matrix
cm_img = f"{results_dir}/confusion_matrix.png"
if os.path.exists(cm_img):
    print("\nConfusion Matrix:")
    display(Image(filename=cm_img, width=600))

# PR curve
pr_img = f"{results_dir}/PR_curve.png"
if os.path.exists(pr_img):
    print("\nPR Curve:")
    display(Image(filename=pr_img, width=600))

# F1 curve
f1_img = f"{results_dir}/F1_curve.png"
if os.path.exists(f1_img):
    print("\nF1 Curve:")
    display(Image(filename=f1_img, width=600))

In [ ]:
DATASET_ROOT = "/content/cn_speed_dataset"

# Run validation on the val set
!python val.py \
    --data "{DATASET_ROOT}/data.yaml" \
    --weights runs/cn_speed_signs/v1/weights/best.pt \
    --img 480 \
    --task val \
    --verbose

## 6. Export ONNX for RKNN

Export with `--rknpu` flag to remove post-processing subgraphs incompatible
with INT8 quantization. This is **required** for the airockchip fork.

Note: Export at 480x480 to match training resolution.

In [ ]:
%cd /content/yolov5

# Export to ONNX with --rknpu flag (required for RKNN conversion)
# [Fix] onnxscript must be installed (done in Environment Setup cell)
!python export.py \
    --weights runs/cn_speed_signs/v1/weights/best.pt \
    --img-size 480 480 \
    --batch-size 1 \
    --rknpu \
    --include onnx

# Verify export
import os
onnx_path = "runs/cn_speed_signs/v1/weights/best.onnx"
if os.path.exists(onnx_path):
    size_mb = os.path.getsize(onnx_path) / 1024 / 1024
    print(f"\nONNX model exported: {onnx_path}")
    print(f"Size: {size_mb:.2f} MB")
else:
    print("ONNX export failed! Check logs above.")
    print("If 'No module named onnxscript', run: !pip install onnxscript")

## 7. Convert to RKNN (INT8 Quantization)

Convert ONNX to RKNN (INT8 quantized) for deployment on RV1106 NPU.

Colab is Linux x86_64, so we CAN run rknn-toolkit2 here. However, Colab's
pre-installed packages (torch 2.10+, numpy 2.x) conflict with rknn-toolkit2's
requirements (torch<=2.4.0, numpy<=1.26.4).

**Solution:** Use an isolated `virtualenv` with its own compatible dependencies.

In [ ]:
%%writefile /content/convert_rknn.py
# ONNX -> RKNN conversion for RV1106 (INT8 quantization)
# CN Speed Signs model (8 classes, 480x480)
import glob, os
from rknn.api import RKNN

ONNX_PATH = "/content/yolov5/runs/cn_speed_signs/v1/weights/best.onnx"
RKNN_PATH = "/content/cn_speed_signs_rv1106.rknn"
DATASET_ROOT = "/content/cn_speed_dataset"

# Prepare calibration images (50 representative training images)
cal_images = sorted(glob.glob(f"{DATASET_ROOT}/train/images/*.jpg"))[:50]
if len(cal_images) < 50:
    cal_images += sorted(glob.glob(f"{DATASET_ROOT}/train/images/*.png"))[:50 - len(cal_images)]

cal_file = "/content/dataset.txt"
with open(cal_file, "w") as f:
    f.write("\n".join(cal_images))
print(f"Calibration images: {len(cal_images)}")

# Initialize RKNN
rknn = RKNN(verbose=False)

# YOLOv5 normalization: input 0-255 -> output 0-1
# mean=[0,0,0], std=[255,255,255] => output = (input - 0) / 255
rknn.config(
    mean_values=[[0, 0, 0]],
    std_values=[[255, 255, 255]],
    target_platform="rv1106",
)

print("Loading ONNX...")
ret = rknn.load_onnx(model=ONNX_PATH)
assert ret == 0, f"Load ONNX failed: {ret}"

print("Building RKNN (INT8 quantization)...")
ret = rknn.build(do_quantization=True, dataset=cal_file)
assert ret == 0, f"Build failed: {ret}"

print("Exporting...")
ret = rknn.export_rknn(RKNN_PATH)
assert ret == 0, f"Export failed: {ret}"

rknn.release()

size_mb = os.path.getsize(RKNN_PATH) / 1024 / 1024
print(f"\nDone! {RKNN_PATH}: {size_mb:.2f} MB")
print(f"Platform: rv1106 (INT8 quantized)")
print(f"Input: 480x480, 8 classes")

In [ ]:
# ============================================================
# Run conversion in isolated virtualenv
#
# [Why virtualenv?]
# Colab has torch 2.10+ and numpy 2.x, but rknn-toolkit2 requires
# torch<=2.4.0 and numpy<=1.26.4. Direct pip install causes
# ContextualVersionConflict at import time, and monkey-patching
# pkg_resources is fragile. A clean virtualenv is the only reliable
# solution.
#
# [Why not venv?]
# Python's built-in venv fails on Colab because ensurepip is missing.
# virtualenv bundles its own pip and works reliably.
#
# [Known dependency fixes]
# - setuptools<70: newer versions removed pkg_resources module
# - onnx==1.16.2: onnx 1.17+ removed onnx.mapping used by rknn-toolkit2
# ============================================================

# Install virtualenv if not present
!pip install virtualenv -q

# Create clean environment and install rknn-toolkit2 with compatible deps
!virtualenv /content/rknn_venv 2>/dev/null || (rm -rf /content/rknn_venv && virtualenv /content/rknn_venv)
!/content/rknn_venv/bin/pip install "setuptools<70" -q
!/content/rknn_venv/bin/pip install "onnx==1.16.2" -q
!/content/rknn_venv/bin/pip install rknn-toolkit2 -q

# Run conversion
!/content/rknn_venv/bin/python3 /content/convert_rknn.py

## 8. Download Trained Model

Download all artifacts for deployment on Luckfox Pico Ultra.

**Important:** The CN model uses 8 classes (vs 9 for AU) and 480x480 input
(vs 320 for AU). You will need to:
1. Update `OBJ_CLASS_NUM` to 8 when building the C binary
2. Update model input size to 480x480 in `rknn_detect.c`
3. Update `class_names[]` in `postprocess.c`

In [ ]:
# Download trained model files
from google.colab import files
import os

print("Downloading model artifacts...")
print("=" * 50)

# PyTorch weights (for future fine-tuning)
pt_path = "/content/yolov5/runs/cn_speed_signs/v1/weights/best.pt"
if os.path.exists(pt_path):
    print(f"  best.pt: {os.path.getsize(pt_path)/1024/1024:.2f} MB")
    files.download(pt_path)

# ONNX model (for RKNN conversion)
onnx_path = "/content/yolov5/runs/cn_speed_signs/v1/weights/best.onnx"
if os.path.exists(onnx_path):
    print(f"  best.onnx: {os.path.getsize(onnx_path)/1024/1024:.2f} MB")
    files.download(onnx_path)

# RKNN model (ready for deployment)
rknn_path = "/content/cn_speed_signs_rv1106.rknn"
if os.path.exists(rknn_path):
    print(f"  cn_speed_signs_rv1106.rknn: {os.path.getsize(rknn_path)/1024/1024:.2f} MB")
    files.download(rknn_path)

print("\n" + "=" * 50)
print("  Deployment Steps:")
print("=" * 50)
print("  1. Copy models to project:")
print("     cp ~/Downloads/best.onnx hub/models/cn_speed_signs.onnx")
print("     cp ~/Downloads/best.pt hub/models/cn_speed_signs.pt")
print("     cp ~/Downloads/cn_speed_signs_rv1106.rknn hub/models/")
print("")
print("  2. Deploy to Luckfox Pico Ultra:")
print("     adb push hub/models/cn_speed_signs_rv1106.rknn /root/model/")
print("")
print("  3. Rebuild ai-hud with CN config:")
print("     cmake -DOBJ_CLASS_NUM=8 ..")
print("     (also update model input to 480x480 and class_names[])")
print("")
print("  4. Run:")
print("     adb shell '/root/ai-hud --model /root/model/cn_speed_signs_rv1106.rknn'")